In [34]:
import os
from pathlib import Path
import librosa
import soundfile as sf
import numpy as np

REAL_INPUT_DIR = "data/realAudio" 
FAKE_INPUT_DIR = "data/fakeAudio"

CLEAN_REAL_DIR = "data/cleanReal" 
CLEAN_FAKE_DIR = "data/cleanFake"

TARGET_SR = 16000
TARGET_DURATION = 3.0


def standardizeAudio(in_path, out_path, sr=TARGET_SR, duration=TARGET_DURATION):
    # Load WAV
    y, _ = librosa.load(in_path, sr=sr, mono=True)
    target_len = int(sr * duration)
    # If too short → pad to center
    if len(y) < target_len:
        pad_total = target_len - len(y)
        pad_left = pad_total // 2
        pad_right = pad_total - pad_left
        y = np.pad(y, (pad_left, pad_right))
    else:
        # Extract the middle 2 seconds
        mid = len(y) // 2
        half = target_len // 2
        start = max(0, mid - half)
        end = start + target_len
        y = y[start:end]
    # Ensure output folder exists
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    # Save standardized WAV
    sf.write(out_path, y, sr)
    print(f"Saved: {out_path}")


def processFolder(input_dir, output_dir):
    input_dir = Path(input_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    audio_files = list(input_dir.glob("*.wav"))
    if not audio_files:
        print(f"No .wav files found in {input_dir}")
        return
    for f in audio_files:
        out_name = f.stem + "Clean.wav"
        out_path = output_dir / out_name
        standardizeAudio(str(f), str(out_path))


def main():
    print("Processing REAL WAV audio")
    processFolder(REAL_INPUT_DIR, CLEAN_REAL_DIR)

    print("\nProcessing FAKE WAV audio")
    processFolder(FAKE_INPUT_DIR, CLEAN_FAKE_DIR)


if __name__ == "__main__":
    main()

Processing REAL WAV audio
Saved: data\cleanReal\audio1Clean.wav
Saved: data\cleanReal\audio10Clean.wav
Saved: data\cleanReal\audio11Clean.wav
Saved: data\cleanReal\audio12Clean.wav
Saved: data\cleanReal\audio13Clean.wav
Saved: data\cleanReal\audio14Clean.wav
Saved: data\cleanReal\audio15Clean.wav
Saved: data\cleanReal\audio16Clean.wav
Saved: data\cleanReal\audio17Clean.wav
Saved: data\cleanReal\audio18Clean.wav
Saved: data\cleanReal\audio19Clean.wav
Saved: data\cleanReal\audio2Clean.wav
Saved: data\cleanReal\audio20Clean.wav
Saved: data\cleanReal\audio3Clean.wav
Saved: data\cleanReal\audio4Clean.wav
Saved: data\cleanReal\audio5Clean.wav
Saved: data\cleanReal\audio6Clean.wav
Saved: data\cleanReal\audio7Clean.wav
Saved: data\cleanReal\audio8Clean.wav
Saved: data\cleanReal\audio9Clean.wav

Processing FAKE WAV audio
Saved: data\cleanFake\audio0Clean.wav
Saved: data\cleanFake\audio1Clean.wav
Saved: data\cleanFake\audio10Clean.wav
Saved: data\cleanFake\audio11Clean.wav
Saved: data\cleanFake\

In [35]:
import os
from pathlib import Path

import librosa
import librosa.display
import numpy as np
import matplotlib.pyplot as plt

REAL_INPUT_DIR = "data/cleanReal"
FAKE_INPUT_DIR = "data/cleanFake"

REAL_OUTPUT_DIR = "data/realSpect"
FAKE_OUTPUT_DIR = "data/fakeSpect"

SAMPLE_RATE = 16000


def create_spectrogram(wav_path, png_path, sr=SAMPLE_RATE):
    # Load audio
    y, sr = librosa.load(wav_path, sr=sr, mono=True)
    # Mel spectrogram
    S = librosa.feature.melspectrogram(
        y=y, sr=sr,
        n_fft=2048,
        hop_length=512,
        n_mels=128
    )
    S_db = librosa.power_to_db(S, ref=np.max)
    # Plot
    plt.figure(figsize=(8, 3))
    librosa.display.specshow(
        S_db, sr=sr, hop_length=512,
        x_axis="time", y_axis="mel"
    )
    plt.colorbar(label="dB")
    plt.title(Path(wav_path).name)
    plt.tight_layout()
    # Save png
    os.makedirs(os.path.dirname(png_path), exist_ok=True)
    plt.savefig(png_path, dpi=150)
    plt.close()

    print(f"Saved: {png_path}")


def processFolder(input_dir, output_dir):
    """Create spectrogram PNGs for all WAV files in a folder."""
    input_dir = Path(input_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    wav_files = sorted(input_dir.glob("*.wav"))
    if not wav_files:
        print(f"No .wav files found in {input_dir}")
        return

    for wav_file in wav_files:
        out_name = wav_file.stem + "_spect.png"
        out_path = output_dir / out_name
        create_spectrogram(str(wav_file), str(out_path))


def main():
    print("Processing cleaned REAL audio")
    processFolder(REAL_INPUT_DIR, REAL_OUTPUT_DIR)

    print("\nProcessing cleaned FAKE audio")
    processFolder(FAKE_INPUT_DIR, FAKE_OUTPUT_DIR)


if __name__ == "__main__":
    main()

Processing cleaned REAL audio
Saved: data\realSpect\audio10Clean_spect.png
Saved: data\realSpect\audio11Clean_spect.png
Saved: data\realSpect\audio12Clean_spect.png
Saved: data\realSpect\audio13Clean_spect.png
Saved: data\realSpect\audio14Clean_spect.png
Saved: data\realSpect\audio15Clean_spect.png
Saved: data\realSpect\audio16Clean_spect.png
Saved: data\realSpect\audio17Clean_spect.png
Saved: data\realSpect\audio18Clean_spect.png
Saved: data\realSpect\audio19Clean_spect.png
Saved: data\realSpect\audio1Clean_spect.png
Saved: data\realSpect\audio20Clean_spect.png
Saved: data\realSpect\audio2Clean_spect.png
Saved: data\realSpect\audio3Clean_spect.png
Saved: data\realSpect\audio4Clean_spect.png
Saved: data\realSpect\audio5Clean_spect.png
Saved: data\realSpect\audio6Clean_spect.png
Saved: data\realSpect\audio7Clean_spect.png
Saved: data\realSpect\audio8Clean_spect.png
Saved: data\realSpect\audio9Clean_spect.png

Processing cleaned FAKE audio
Saved: data\fakeSpect\audio0Clean_spect.png
Save

In [36]:
import os
import glob
import numpy as np
import pandas as pd
import librosa

def extractFingerprint(filepath, sr=16000, n_mfcc=13):
    # Load audio 
    y, sr = librosa.load(filepath, sr=sr, mono=True)
    features = {}

    # MFCC
    mfcc = librosa.feature.mfcc(
        y=y, sr=sr, n_mfcc=n_mfcc,
        dct_type=2, norm='ortho'   # for stable reproducibility
    )

    for i in range(n_mfcc):
        features[f"mfcc_{i+1}_mean"] = float(mfcc[i].mean())
        features[f"mfcc_{i+1}_std"] = float(mfcc[i].std())

    # Spectral features
    centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
    bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)
    rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr, roll_percent=0.95)
    flatness = librosa.feature.spectral_flatness(y=y)

    features["spect_centroid_mean"] = float(centroid.mean())
    features["spect_centroid_std"] = float(centroid.std())

    features["spect_bandwidth_mean"] = float(bandwidth.mean())
    features["spect_bandwidth_std"] = float(bandwidth.std())

    features["spect_rolloff95_mean"] = float(rolloff.mean())
    features["spect_rolloff95_std"] = float(rolloff.std())

    features["spect_flatness_mean"] = float(flatness.mean())
    features["spect_flatness_std"] = float(flatness.std())

    # Temporal / energy
    zcr = librosa.feature.zero_crossing_rate(y)
    rms = librosa.feature.rms(y=y)
    flux = librosa.onset.onset_strength(y=y, sr=sr)

    features["zcr_mean"] = float(zcr.mean())
    features["zcr_std"] = float(zcr.std())

    features["rms_mean"] = float(rms.mean())
    features["rms_std"] = float(rms.std())

    features["spec_flux_mean"] = float(flux.mean())
    features["spec_flux_std"] = float(flux.std())

    return features


def processFolders(real_dir, fake_dir):
    rows = []
    # Real
    for filepath in glob.glob(os.path.join(real_dir, "*.wav")):
        print(f"Processing (real): {filepath}")
        feats = extractFingerprint(filepath)
        feats["label"] = "real"
        feats["file"] = os.path.basename(filepath)
        rows.append(feats)

    # Fake
    for filepath in glob.glob(os.path.join(fake_dir, "*.wav")):
        print(f"Processing (deepfake): {filepath}")
        feats = extractFingerprint(filepath)
        feats["label"] = "deepfake"
        feats["file"] = os.path.basename(filepath)
        rows.append(feats)

    return pd.DataFrame(rows)


if __name__ == "__main__":
    REAL_DIR = "data/cleanReal"
    FAKE_DIR = "data/cleanFake"
    OUTPUT_CSV = "data/spectralFingerprints.csv"

    df = processFolders(REAL_DIR, FAKE_DIR)
    print("Extracted features shape:", df.shape)
    print(df.head())

    df.to_csv(OUTPUT_CSV, index=False)
    print(f"Saved features to {OUTPUT_CSV}")

Processing (real): data/cleanReal\audio10Clean.wav
Processing (real): data/cleanReal\audio11Clean.wav
Processing (real): data/cleanReal\audio12Clean.wav
Processing (real): data/cleanReal\audio13Clean.wav
Processing (real): data/cleanReal\audio14Clean.wav
Processing (real): data/cleanReal\audio15Clean.wav
Processing (real): data/cleanReal\audio16Clean.wav
Processing (real): data/cleanReal\audio17Clean.wav
Processing (real): data/cleanReal\audio18Clean.wav
Processing (real): data/cleanReal\audio19Clean.wav
Processing (real): data/cleanReal\audio1Clean.wav
Processing (real): data/cleanReal\audio20Clean.wav
Processing (real): data/cleanReal\audio2Clean.wav
Processing (real): data/cleanReal\audio3Clean.wav
Processing (real): data/cleanReal\audio4Clean.wav
Processing (real): data/cleanReal\audio5Clean.wav
Processing (real): data/cleanReal\audio6Clean.wav
Processing (real): data/cleanReal\audio7Clean.wav
Processing (real): data/cleanReal\audio8Clean.wav
Processing (real): data/cleanReal\audio

**This is the to determine the suitable number of PCA components.**

In [37]:
'''import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import numpy as np

CSV_PATH = "data/spectralFingerprints.csv"

def find_optimal_pca(csv_path=CSV_PATH, variance_threshold=0.95):
    # Load data
    df = pd.read_csv(csv_path)

    # Separate features
    X = df.drop(columns=["label", "file"])

    # Standardize
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # PCA with all components
    pca_full = PCA()
    pca_full.fit(X_scaled)

    # Explained variance
    explained = pca_full.explained_variance_ratio_
    cumulative = explained.cumsum()

    # Find smallest k such that cumulative variance >= threshold
    k_opt = np.argmax(cumulative >= variance_threshold) + 1

    print("\nExplained variance ratio (per component):")
    print(explained)
    print("\nCumulative explained variance:")
    print(cumulative)
    print(f"\nOptimal number of components for {variance_threshold} variance: {k_opt}")

    return k_opt, cumulative


if __name__ == "__main__":
    k_opt, cumulative = find_optimal_pca()'''

'import pandas as pd\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.decomposition import PCA\nimport matplotlib.pyplot as plt\nimport numpy as np\n\nCSV_PATH = "data/spectralFingerprints.csv"\n\ndef find_optimal_pca(csv_path=CSV_PATH, variance_threshold=0.95):\n    # Load data\n    df = pd.read_csv(csv_path)\n\n    # Separate features\n    X = df.drop(columns=["label", "file"])\n\n    # Standardize\n    scaler = StandardScaler()\n    X_scaled = scaler.fit_transform(X)\n\n    # PCA with all components\n    pca_full = PCA()\n    pca_full.fit(X_scaled)\n\n    # Explained variance\n    explained = pca_full.explained_variance_ratio_\n    cumulative = explained.cumsum()\n\n    # Find smallest k such that cumulative variance >= threshold\n    k_opt = np.argmax(cumulative >= variance_threshold) + 1\n\n    print("\nExplained variance ratio (per component):")\n    print(explained)\n    print("\nCumulative explained variance:")\n    print(cumulative)\n    print(f"\nOptimal number

**This is the PCA algorithm and it's effect on the code. Accuracy with PCA + KNN = 0.73 (decrease from 0.86).**

In [38]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import numpy as np

CSV_PATH = "data/spectralFingerprints.csv"

def run_pca(csv_path=CSV_PATH, n_components=17, top_features=5):
    # Load dataset
    df = pd.read_csv(csv_path)

    # Separate features and labels
    X = df.drop(columns=["label", "file"])
    y = df["label"]
    file_names = df["file"]

    feature_names = X.columns.tolist()

    # Standardize features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Apply PCA
    pca = PCA(n_components=n_components)
    X_pca = pca.fit_transform(X_scaled)

    # Display explained variance
    print("\nExplained Variance Ratio")
    for i, var in enumerate(pca.explained_variance_ratio_):
        print(f"PC{i+1}: {var:.4f}")

    print("\nTOP CONTRIBUTING FEATURES PER PCA COMPONENT")
    loadings = pca.components_ 

    for i in range(n_components):
        component = loadings[i]
        # Sort features by absolute loading strength
        sorted_idx = np.argsort(np.abs(component))[::-1]
        top_idx = sorted_idx[:top_features]

        print(f"\nTop {top_features} features contributing to PC{i+1}:")
        for idx in top_idx:
            print(f"{feature_names[idx]} (loading={component[idx]:.4f})")

    # -----------------------------------------------------------
    # Build final PCA dataset
    # -----------------------------------------------------------
    df_final = pd.DataFrame(X_pca, columns=[f"PC{i+1}" for i in range(n_components)])
    df_final["label"] = y
    df_final["file"] = file_names

    return df_final, pca


if __name__ == "__main__":
    df_final, pca_model = run_pca()
    print("\nPCA-Reduced DataFrame:")
    print(df_final.head())


Explained Variance Ratio
PC1: 0.2440
PC2: 0.2126
PC3: 0.0937
PC4: 0.0857
PC5: 0.0653
PC6: 0.0480
PC7: 0.0360
PC8: 0.0306
PC9: 0.0258
PC10: 0.0224
PC11: 0.0192
PC12: 0.0150
PC13: 0.0129
PC14: 0.0123
PC15: 0.0110
PC16: 0.0099
PC17: 0.0092

TOP CONTRIBUTING FEATURES PER PCA COMPONENT

Top 5 features contributing to PC1:
rms_std (loading=0.2517)
mfcc_2_mean (loading=-0.2363)
mfcc_2_std (loading=0.2337)
rms_mean (loading=0.2328)
zcr_mean (loading=0.2313)

Top 5 features contributing to PC2:
spect_flatness_mean (loading=0.3079)
spect_flatness_std (loading=0.2832)
spect_bandwidth_std (loading=0.2785)
spect_bandwidth_mean (loading=-0.2743)
spec_flux_mean (loading=-0.2706)

Top 5 features contributing to PC3:
mfcc_11_mean (loading=0.3442)
mfcc_10_std (loading=0.3144)
mfcc_10_mean (loading=-0.3062)
mfcc_7_std (loading=0.2992)
mfcc_7_mean (loading=-0.2874)

Top 5 features contributing to PC4:
mfcc_5_mean (loading=0.4577)
mfcc_3_mean (loading=0.3698)
mfcc_10_mean (loading=0.3535)
mfcc_7_mean (loa

**This is the OPTICS algorithm and it's effect on the code. Accuracy with OPTICS + KNN = 0.8 (decrease from 0.86).**

In [39]:
"""from sklearn.cluster import OPTICS
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np

CSV_PATH = "data/spectralFingerprints.csv"

# Load data
df = pd.read_csv(CSV_PATH)

# Separate features and non-features
X = df.drop(columns=["label", "file"])
y = df["label"]

# SCALE FEATURES
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Run OPTICS on scaled data
optics = OPTICS(
    max_eps=8,
    min_samples=2,
    metric='euclidean',
    min_cluster_size=2
)

optics.fit(X_scaled)

labels = optics.labels_
df["cluster"] = labels

# Check if any real clusters exist
non_noise_labels = [lab for lab in labels if lab != -1]

if len(non_noise_labels) == 0:
    print("\n[WARNING] OPTICS FOUND NO CLUSTERS — ALL POINTS LABELED -1")
    print("OPTICS is NOT removing outliers. Returning full dataset unchanged.")
    # Use full df as final result (just drop cluster column)
    df_final = df.drop(columns=["cluster"])

else:
    # Identify outliers
    indices_to_drop = df[df["cluster"] == -1].index
    print("\nOUTLIER INDICES TO DROP (-1 labels):")
    print(indices_to_drop.tolist())

    print(f"\nOutliers detected: {len(indices_to_drop)}")
    print(f"Total samples: {len(df)}")
    print(f"Remaining samples: {len(df) - len(indices_to_drop)}")

    # Drop outliers
    df_new = df.drop(indices_to_drop).reset_index(drop=True)

    # Final cleaned dataset
    df_final = df_new.drop(columns=["cluster"])"""

'from sklearn.cluster import OPTICS\nfrom sklearn.preprocessing import StandardScaler\nimport pandas as pd\nimport numpy as np\n\nCSV_PATH = "data/spectralFingerprints.csv"\n\n# Load data\ndf = pd.read_csv(CSV_PATH)\n\n# Separate features and non-features\nX = df.drop(columns=["label", "file"])\ny = df["label"]\n\n# SCALE FEATURES\nscaler = StandardScaler()\nX_scaled = scaler.fit_transform(X)\n\n# Run OPTICS on scaled data\noptics = OPTICS(\n    max_eps=8,\n    min_samples=2,\n    metric=\'euclidean\',\n    min_cluster_size=2\n)\n\noptics.fit(X_scaled)\n\nlabels = optics.labels_\ndf["cluster"] = labels\n\n# Check if any real clusters exist\nnon_noise_labels = [lab for lab in labels if lab != -1]\n\nif len(non_noise_labels) == 0:\n    print("\n[WARNING] OPTICS FOUND NO CLUSTERS — ALL POINTS LABELED -1")\n    print("OPTICS is NOT removing outliers. Returning full dataset unchanged.")\n    # Use full df as final result (just drop cluster column)\n    df_final = df.drop(columns=["cluster"]

**This is the isolation forest that gives us the outliers. By removing said outliers, we were able to get an improved accuracy of 0.91 (compared to 0.86)**

In [40]:
"""from sklearn.ensemble import IsolationForest
import pandas as pd
import numpy as np

CSV_PATH = "data/spectralFingerprints.csv"
df = pd.read_csv(CSV_PATH)

# Separate features/labels
X = df.drop(columns=["label", "file"])
y = df["label"]

clf = IsolationForest( random_state=0)

outlier_flags = clf.fit_predict(X)   # -1 = outlier, 1 = inlier
df['outlier'] = outlier_flags

# Summary counts\
n_outliers = np.sum(outlier_flags == -1)
n_inliers = np.sum(outlier_flags == 1)

print("OUTLIER STATISTICS")
print(f"Total outliers detected: {n_outliers}")
print(f"Total inliers: {n_inliers}")

# Show which indices will be dropped
indices_to_drop = df[df['outlier'] == -1].index
print("OUTLIER INDICES")
print(indices_to_drop.tolist())

# Drop outliers
df_new = df[df['outlier'] == 1].reset_index(drop=True)

# Final cleaned dataset
df_final = df_new.drop(columns=["outlier"])"""

'from sklearn.ensemble import IsolationForest\nimport pandas as pd\nimport numpy as np\n\nCSV_PATH = "data/spectralFingerprints.csv"\ndf = pd.read_csv(CSV_PATH)\n\n# Separate features/labels\nX = df.drop(columns=["label", "file"])\ny = df["label"]\n\nclf = IsolationForest( random_state=0)\n\noutlier_flags = clf.fit_predict(X)   # -1 = outlier, 1 = inlier\ndf[\'outlier\'] = outlier_flags\n\n# Summary countsn_outliers = np.sum(outlier_flags == -1)\nn_inliers = np.sum(outlier_flags == 1)\n\nprint("OUTLIER STATISTICS")\nprint(f"Total outliers detected: {n_outliers}")\nprint(f"Total inliers: {n_inliers}")\n\n# Show which indices will be dropped\nindices_to_drop = df[df[\'outlier\'] == -1].index\nprint("OUTLIER INDICES")\nprint(indices_to_drop.tolist())\n\n# Drop outliers\ndf_new = df[df[\'outlier\'] == 1].reset_index(drop=True)\n\n# Final cleaned dataset\ndf_final = df_new.drop(columns=["outlier"])'

**KNN training. If you wish to run this on the orginal data, change 'df_final' to 'df'.**

In [41]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import Normalizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import LeaveOneOut

# Path to your CSV from the feature-extraction script
CSV_PATH = "data/spectralFingerprints.csv"

def main():
    # Load data
    df = pd.read_csv(CSV_PATH)

    
    # Drop non-feature columns
    X = df_final.drop(columns=["label", "file"])
    y = df_final["label"]

    loo = LeaveOneOut()
    print(loo.get_n_splits(X))
    avg_result = 0
    for i, (train_index, test_index) in enumerate(loo.split(X)):
        #print(f"Fold {i}:")
        #print(f"  Train: index={train_index}")
        #print(f"  Test:  index={test_index}")

        X_train = X.iloc[train_index]
        y_train = y.iloc[train_index]
        X_test = X.iloc[test_index]
        y_test = y.iloc[test_index]

        clf = Pipeline([
        ("scaler", StandardScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=5))
        ])

        # Train
        clf.fit(X_train, y_train)

        # Evaluate
        acc = clf.score(X_test, y_test)
        #print(f"Accuracy: {acc:.4f}\n")
        avg_result += acc
    print('Avg. Accuracy with every datapoint as the test sample:')
    print(avg_result/loo.get_n_splits(X))

    # Train/test split
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.3,
        random_state=42,
        stratify=y
    )

    # Pipeline: standardize features -> KNN
    clf = Pipeline([
        ("scaler", StandardScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=5))
    ])

    # Train
    clf.fit(X_train, y_train)

    print("\nCross validation score:")
    print(cross_val_score(clf, X_train, y_train))
    print()

    # Evaluate
    acc = clf.score(X_test, y_test)
    print(f"Accuracy: {acc:.4f}\n")

    y_pred = clf.predict(X_test)

    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

if __name__ == "__main__":
    main()

50
Avg. Accuracy with every datapoint as the test sample:
0.86

Cross validation score:
[0.85714286 0.71428571 0.85714286 0.85714286 0.57142857]

Accuracy: 0.8667

Confusion Matrix:
[[7 2]
 [0 6]]

Classification Report:
              precision    recall  f1-score   support

    deepfake       1.00      0.78      0.88         9
        real       0.75      1.00      0.86         6

    accuracy                           0.87        15
   macro avg       0.88      0.89      0.87        15
weighted avg       0.90      0.87      0.87        15



In [42]:
import os
import glob
import numpy as np
from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import Normalizer
from sklearn.preprocessing import MinMaxScaler

def load_spectrogram(filepath, size=(128, 128)):
    """
    Load a PNG spectrogram, convert to grayscale, resize,
    normalize, and flatten to 1D.
    """
    img = Image.open(filepath).convert("L")  # grayscale
    img = img.resize(size)                  # enforce fixed size
    arr = np.array(img, dtype=np.float32)

    # normalize to [0,1]
    arr /= 255.0

    # flatten into a 1D feature vector
    return arr.flatten()


def load_spectrogram_dataset(real_dir, fake_dir, img_size=(128,128)):
    X, y, files = [], [], []

    # Real spectrograms
    for filepath in glob.glob(os.path.join(real_dir, "*.png")):
        feat = load_spectrogram(filepath, size=img_size)
        X.append(feat)
        y.append("real")
        files.append(os.path.basename(filepath))

    # Fake spectrograms
    for filepath in glob.glob(os.path.join(fake_dir, "*.png")):
        feat = load_spectrogram(filepath, size=img_size)
        X.append(feat)
        y.append("deepfake")
        files.append(os.path.basename(filepath))

    X = np.vstack(X)
    y = np.array(y)
    files = np.array(files)

    print("Dataset loaded.")
    print("Feature matrix shape:", X.shape)
    print("Class counts:")
    print("  real:", np.sum(y=="real"))
    print("  deepfake:", np.sum(y=="deepfake"))

    return X, y, files


if __name__ == "__main__":
    REAL_SPECT_DIR = "data/realSpect"
    FAKE_SPECT_DIR = "data/fakeSpect"

    # Load data
    X, y, files = load_spectrogram_dataset(REAL_SPECT_DIR, FAKE_SPECT_DIR)

    # Split dataset
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.3,
        random_state=42,
        stratify=y
    )

    # KNN pipeline with StandardScaler
    clf = Pipeline([
        ("scaler", StandardScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=5))
    ])

    clf.fit(X_train, y_train)
    
    print("\nCross validation score:")
    print(cross_val_score(clf, X_train, y_train))
    print()

    # Evaluate
    acc = clf.score(X_test, y_test)
    print(f"\nAccuracy: {acc:.4f}")

    y_pred = clf.predict(X_test)

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred, labels=["real", "deepfake"]))

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, labels=["real", "deepfake"]))

Dataset loaded.
Feature matrix shape: (50, 16384)
Class counts:
  real: 20
  deepfake: 30

Cross validation score:
[0.71428571 0.57142857 1.         0.42857143 0.71428571]


Accuracy: 0.8667

Confusion Matrix:
[[5 1]
 [1 8]]

Classification Report:
              precision    recall  f1-score   support

        real       0.83      0.83      0.83         6
    deepfake       0.89      0.89      0.89         9

    accuracy                           0.87        15
   macro avg       0.86      0.86      0.86        15
weighted avg       0.87      0.87      0.87        15

